In [1]:
import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS

In [2]:
PANEL_FILE = "transaction_panel_v6_clean.xlsx"
 
L = 7 # distributed-lag window in days
 
ENTITY = "asa_id"
DATE   = "date"
 
df = pd.read_excel(PANEL_FILE)
df[DATE] = pd.to_datetime(df[DATE])
df = df.sort_values([ENTITY, DATE])
print(f"Loaded: {df.shape[0]:,} rows, {df[ENTITY].nunique()} entities")

Loaded: 64,194 rows, 192 entities


# Outflows 

In [3]:
X1 = [
    "delta_vix_z",
    "delta_dgs10_z",
    "delta_dgs1mo_z",
    "ads_z",
    "eth_return_z",
    "vnq_return_z",
    "mortgage_ret_z",
    "eth_shock",
]

Y = "log_outflow"

print("=" * 65)
print("  Model 1: All Variables")
print("=" * 65)
 
data_m1 = (
    df[[ENTITY, DATE, Y] + X1]
    .dropna()
    .set_index([ENTITY, DATE])
)
print(f"N obs: {len(data_m1):,}  |  N entities: {data_m1.index.get_level_values(0).nunique()}\n")
 
m1   = PanelOLS(data_m1[Y], data_m1[X1], entity_effects=True, time_effects=False)
res1 = m1.fit(cov_type="clustered", cluster_entity=True, cluster_time=True)
print(res1.summary)
 
print("=" * 65)
print("  Model 1b: All Variables (Driscoll-Kraay)")
print("=" * 65)
 
m1b   = PanelOLS(data_m1[Y], data_m1[X1], entity_effects=True, time_effects=False)
res1b = m1b.fit(cov_type="kernel", kernel="bartlett", bandwidth=7)
print(res1b.summary)

  Model 1: All Variables
N obs: 64,194  |  N entities: 192

                          PanelOLS Estimation Summary                           
Dep. Variable:            log_outflow   R-squared:                        0.0031
Estimator:                   PanelOLS   R-squared (Between):             -0.0259
No. Observations:               64194   R-squared (Within):               0.0031
Date:                Wed, May 27 2026   R-squared (Overall):             -0.0178
Time:                        21:19:03   Log-likelihood                -1.166e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      24.821
Entities:                         192   P-value                           0.0000
Avg Obs:                       334.34   Distribution:                 F(8,63994)
Min Obs:                       5.0000                                           
Max Obs:                       1137.0   F-statist

In [4]:
print("=" * 65)
print(f"  Model 2: Distributed-Lag FE (L={L}) — clustered (entity + time)")
print("=" * 65)
 
df_dl  = df[[ENTITY, DATE, Y] + X1].dropna().copy()
 
date_x = (
    df_dl[[DATE] + X1]
    .drop_duplicates(subset=DATE)
    .sort_values(DATE)
    .reset_index(drop=True)
    .copy()
)
 
X2 = []
for var in X1:
    for lag in range(L + 1):
        lag_name = f"{var}_lag{lag}"
        date_x[lag_name] = date_x[var].shift(lag)
        X2.append(lag_name)
 
data_dl = (
    df_dl[[ENTITY, DATE, Y]]
    .merge(date_x[[DATE] + X2], on=DATE, how="left")
    .dropna(subset=[Y] + X2)
    .set_index([ENTITY, DATE])
    .sort_index()
)
print(f"N obs: {len(data_dl):,}  |  N entities: {data_dl.index.get_level_values(0).nunique()}\n")
 
m2   = PanelOLS(data_dl[Y], data_dl[X2], entity_effects=True, time_effects=False)
res2 = m2.fit(cov_type="clustered", cluster_entity=True, cluster_time=True)
print(res2.summary)
 
print("=" * 65)
print(f"  Model 2b: Distributed-Lag FE (L={L}) — Driscoll-Kraay (Bartlett, bw=7)")
print("=" * 65)
 
m2b   = PanelOLS(data_dl[Y], data_dl[X2], entity_effects=True, time_effects=False)
res2b = m2b.fit(cov_type="kernel", kernel="bartlett", bandwidth=7)
print(res2b.summary)

  Model 2: Distributed-Lag FE (L=7) — clustered (entity + time)
N obs: 64,187  |  N entities: 192

                          PanelOLS Estimation Summary                           
Dep. Variable:            log_outflow   R-squared:                        0.0116
Estimator:                   PanelOLS   R-squared (Between):             -0.1121
No. Observations:               64187   R-squared (Within):               0.0116
Date:                Wed, May 27 2026   R-squared (Overall):             -0.0713
Time:                        21:19:05   Log-likelihood                -1.163e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      11.748
Entities:                         192   P-value                           0.0000
Avg Obs:                       334.31   Distribution:                F(64,63931)
Min Obs:                       5.0000                                           
Max Obs:  

In [5]:
X3 = [
    "delta_vix_z",
    "delta_dgs10_z",
    "delta_dgs1mo_z",
    "ads_z",
    "eth_return_z",
    "mortgage_ret_z",
    "eth_shock",
]

print("=" * 65)
print("  Model 3: VNQ excluded")
print("=" * 65)
 
data_m3 = (
    df[[ENTITY, DATE, Y] + X3]
    .dropna()
    .set_index([ENTITY, DATE])
)
 
m3   = PanelOLS(data_m3[Y], data_m3[X3], entity_effects=True, time_effects=False)
res3 = m3.fit(cov_type="clustered", cluster_entity=True, cluster_time=True)
print(res3.summary)
 
print("=" * 65)
print("  Model 3b: VNQ excluded (Driscoll-Kraay)")
print("=" * 65)
 
m3b   = PanelOLS(data_m3[Y], data_m3[X3], entity_effects=True, time_effects=False)
res3b = m3b.fit(cov_type="kernel", kernel="bartlett", bandwidth=7)
print(res3b.summary)

  Model 3: VNQ excluded
                          PanelOLS Estimation Summary                           
Dep. Variable:            log_outflow   R-squared:                        0.0031
Estimator:                   PanelOLS   R-squared (Between):             -0.0259
No. Observations:               64194   R-squared (Within):               0.0031
Date:                Wed, May 27 2026   R-squared (Overall):             -0.0178
Time:                        21:19:08   Log-likelihood                -1.166e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      28.363
Entities:                         192   P-value                           0.0000
Avg Obs:                       334.34   Distribution:                 F(7,63995)
Min Obs:                       5.0000                                           
Max Obs:                       1137.0   F-statistic (robust):             4.2645
    

In [6]:
print("=" * 65)
print(f"  Model 4: Distributed-Lag FE (L={L}) — clustered (entity + time)")
print("=" * 65)

df_dl  = df[[ENTITY, DATE, Y] + X3].dropna().copy()
 
date_x = (
    df_dl[[DATE] + X3]
    .drop_duplicates(subset=DATE)
    .sort_values(DATE)
    .reset_index(drop=True)
    .copy()
)
 
X4 = []
for var in X3:
    for lag in range(L + 1):
        lag_name = f"{var}_lag{lag}"
        date_x[lag_name] = date_x[var].shift(lag)
        X4.append(lag_name)
 
data_dl = (
    df_dl[[ENTITY, DATE, Y]]
    .merge(date_x[[DATE] + X4], on=DATE, how="left")
    .dropna(subset=[Y] + X4)
    .set_index([ENTITY, DATE])
    .sort_index()
)
 
m4   = PanelOLS(data_dl[Y], data_dl[X4], entity_effects=True, time_effects=False)
res4 = m4.fit(cov_type="clustered", cluster_entity=True, cluster_time=True)
print(res4.summary)
 
print("=" * 65)
print(f"  Model 4b: Distributed-Lag FE (L={L}) — Driscoll-Kraay (Bartlett, bw=7)")
print("=" * 65)
 
m4b   = PanelOLS(data_dl[Y], data_dl[X4], entity_effects=True, time_effects=False)
res4b = m4b.fit(cov_type="kernel", kernel="bartlett", bandwidth=7)
print(res4b.summary)

  Model 4: Distributed-Lag FE (L=7) — clustered (entity + time)
                          PanelOLS Estimation Summary                           
Dep. Variable:            log_outflow   R-squared:                        0.0101
Estimator:                   PanelOLS   R-squared (Between):             -0.1074
No. Observations:               64187   R-squared (Within):               0.0101
Date:                Wed, May 27 2026   R-squared (Overall):             -0.0703
Time:                        21:19:13   Log-likelihood                -1.164e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      11.670
Entities:                         192   P-value                           0.0000
Avg Obs:                       334.31   Distribution:                F(56,63939)
Min Obs:                       5.0000                                           
Max Obs:                       1137.0   F-sta

In [7]:
X5 = [
    "delta_vix_z",
    "delta_dgs10_z",
    "ads_z",
    "eth_return_z",
    "mortgage_ret_z",
    "eth_shock",
]

print("=" * 65)
print("  Model 5: VNQ + 1 month treasury excluded")
print("=" * 65)
 
data_m5 = (
    df[[ENTITY, DATE, Y] + X5]
    .dropna()
    .set_index([ENTITY, DATE])
)
 
m5   = PanelOLS(data_m3[Y], data_m3[X5], entity_effects=True, time_effects=False)
res5 = m5.fit(cov_type="clustered", cluster_entity=True, cluster_time=True)
print(res5.summary)
 
print("=" * 65)
print("  Model 5b: VNQ + 1 month treasury (Driscoll-Kraay)")
print("=" * 65)
 
m5b   = PanelOLS(data_m5[Y], data_m5[X5], entity_effects=True, time_effects=False)
res5b = m5b.fit(cov_type="kernel", kernel="bartlett", bandwidth=7)
print(res5b.summary)

  Model 5: VNQ + 1 month treasury excluded
                          PanelOLS Estimation Summary                           
Dep. Variable:            log_outflow   R-squared:                        0.0031
Estimator:                   PanelOLS   R-squared (Between):             -0.0259
No. Observations:               64194   R-squared (Within):               0.0031
Date:                Wed, May 27 2026   R-squared (Overall):             -0.0178
Time:                        21:19:15   Log-likelihood                -1.166e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      33.064
Entities:                         192   P-value                           0.0000
Avg Obs:                       334.34   Distribution:                 F(6,63996)
Min Obs:                       5.0000                                           
Max Obs:                       1137.0   F-statistic (robust):     

In [8]:
print("=" * 65)
print(f"  Model 6: Distributed-Lag FE (L={L}) — clustered (entity + time)")
print("=" * 65)

df_dl  = df[[ENTITY, DATE, Y] + X5].dropna().copy()
 
date_x = (
    df_dl[[DATE] + X5]
    .drop_duplicates(subset=DATE)
    .sort_values(DATE)
    .reset_index(drop=True)
    .copy()
)
 
X6 = []
for var in X5:
    for lag in range(L + 1):
        lag_name = f"{var}_lag{lag}"
        date_x[lag_name] = date_x[var].shift(lag)
        X6.append(lag_name)
 
data_dl = (
    df_dl[[ENTITY, DATE, Y]]
    .merge(date_x[[DATE] + X6], on=DATE, how="left")
    .dropna(subset=[Y] + X6)
    .set_index([ENTITY, DATE])
    .sort_index()
)
 
m6   = PanelOLS(data_dl[Y], data_dl[X6], entity_effects=True, time_effects=False)
res6 = m6.fit(cov_type="clustered", cluster_entity=True, cluster_time=True)
print(res6.summary)
 
print("=" * 65)
print(f"  Model 6b: Distributed-Lag FE (L={L}) — Driscoll-Kraay (Bartlett, bw=7)")
print("=" * 65)
 
m6b   = PanelOLS(data_dl[Y], data_dl[X6], entity_effects=True, time_effects=False)
res6b = m6b.fit(cov_type="kernel", kernel="bartlett", bandwidth=7)
print(res6b.summary)

  Model 6: Distributed-Lag FE (L=7) — clustered (entity + time)
                          PanelOLS Estimation Summary                           
Dep. Variable:            log_outflow   R-squared:                        0.0099
Estimator:                   PanelOLS   R-squared (Between):             -0.1080
No. Observations:               64187   R-squared (Within):               0.0099
Date:                Wed, May 27 2026   R-squared (Overall):             -0.0708
Time:                        21:19:17   Log-likelihood                -1.164e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      13.275
Entities:                         192   P-value                           0.0000
Avg Obs:                       334.31   Distribution:                F(48,63947)
Min Obs:                       5.0000                                           
Max Obs:                       1137.0   F-sta

In [9]:
X7 = [
    "delta_vix_z",
    "delta_dgs10_z",
    "ads_z",
    "eth_return_z",
    "eth_shock",
]

print("=" * 65)
print("  Model 7: VNQ, 1 month treasury + Mortgage Index Returns excluded")
print("=" * 65)
 
data_m7 = (
    df[[ENTITY, DATE, Y] + X7]
    .dropna()
    .set_index([ENTITY, DATE])
)
 
m7   = PanelOLS(data_m7[Y], data_m7[X7], entity_effects=True, time_effects=False)
res7 = m7.fit(cov_type="clustered", cluster_entity=True, cluster_time=True)
print(res7.summary)
 
print("=" * 65)
print("  Model 7b: VNQ, 1 month treasury + Mortgage Index Returns excluded (Driscoll-Kraay)")
print("=" * 65)
 
m7b   = PanelOLS(data_m7[Y], data_m7[X7], entity_effects=True, time_effects=False)
res7b = m7b.fit(cov_type="kernel", kernel="bartlett", bandwidth=7)
print(res7b.summary)

  Model 7: VNQ, 1 month treasury + Mortgage Index Returns excluded
                          PanelOLS Estimation Summary                           
Dep. Variable:            log_outflow   R-squared:                        0.0031
Estimator:                   PanelOLS   R-squared (Between):             -0.0258
No. Observations:               64194   R-squared (Within):               0.0031
Date:                Wed, May 27 2026   R-squared (Overall):             -0.0178
Time:                        21:19:18   Log-likelihood                -1.166e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      39.400
Entities:                         192   P-value                           0.0000
Avg Obs:                       334.34   Distribution:                 F(5,63997)
Min Obs:                       5.0000                                           
Max Obs:                       1137.0   F-

In [10]:
print("=" * 65)
print(f"  Model 8: Distributed-Lag FE (L={L}) — clustered (entity + time)")
print("=" * 65)

df_dl  = df[[ENTITY, DATE, Y] + X7].dropna().copy()
 
date_x = (
    df_dl[[DATE] + X7]
    .drop_duplicates(subset=DATE)
    .sort_values(DATE)
    .reset_index(drop=True)
    .copy()
)
 
X8 = []
for var in X7:
    for lag in range(L + 1):
        lag_name = f"{var}_lag{lag}"
        date_x[lag_name] = date_x[var].shift(lag)
        X8.append(lag_name)
 
data_dl = (
    df_dl[[ENTITY, DATE, Y]]
    .merge(date_x[[DATE] + X8], on=DATE, how="left")
    .dropna(subset=[Y] + X8)
    .set_index([ENTITY, DATE])
    .sort_index()
)
 
m8   = PanelOLS(data_dl[Y], data_dl[X8], entity_effects=True, time_effects=False)
res8 = m8.fit(cov_type="clustered", cluster_entity=True, cluster_time=True)
print(res8.summary)
 
print("=" * 65)
print(f"  Model 8b: Distributed-Lag FE (L={L}) — Driscoll-Kraay (Bartlett, bw=7)")
print("=" * 65)
 
m8b   = PanelOLS(data_dl[Y], data_dl[X8], entity_effects=True, time_effects=False)
res8b = m8b.fit(cov_type="kernel", kernel="bartlett", bandwidth=7)
print(res8b.summary)

  Model 8: Distributed-Lag FE (L=7) — clustered (entity + time)
                          PanelOLS Estimation Summary                           
Dep. Variable:            log_outflow   R-squared:                        0.0090
Estimator:                   PanelOLS   R-squared (Between):             -0.1062
No. Observations:               64187   R-squared (Within):               0.0090
Date:                Wed, May 27 2026   R-squared (Overall):             -0.0696
Time:                        21:19:19   Log-likelihood                -1.164e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      14.540
Entities:                         192   P-value                           0.0000
Avg Obs:                       334.31   Distribution:                F(40,63955)
Min Obs:                       5.0000                                           
Max Obs:                       1137.0   F-sta

In [11]:
X9 = [
    "delta_dgs10_z",
    "ads_z",
    "eth_return_z",
    "eth_shock",
]

print("=" * 65)
print("  Model 9: VNQ, 1 month treasury, Mortgage Index Returns + VIX excluded")
print("=" * 65)
 
data_m9 = (
    df[[ENTITY, DATE, Y] + X9]
    .dropna()
    .set_index([ENTITY, DATE])
)
 
m9   = PanelOLS(data_m9[Y], data_m9[X9], entity_effects=True, time_effects=False)
res9 = m9.fit(cov_type="clustered", cluster_entity=True, cluster_time=True)
print(res9.summary)
 
print("=" * 65)
print("  Model 9b: VNQ, 1 month treasury, Mortgage Index Returns + VIX excluded (Driscoll-Kraay)")
print("=" * 65)
 
m9b   = PanelOLS(data_m9[Y], data_m9[X9], entity_effects=True, time_effects=False)
res9b = m9b.fit(cov_type="kernel", kernel="bartlett", bandwidth=7)
print(res9b.summary)

  Model 9: VNQ, 1 month treasury, Mortgage Index Returns + VIX excluded
                          PanelOLS Estimation Summary                           
Dep. Variable:            log_outflow   R-squared:                        0.0031
Estimator:                   PanelOLS   R-squared (Between):             -0.0256
No. Observations:               64194   R-squared (Within):               0.0031
Date:                Wed, May 27 2026   R-squared (Overall):             -0.0176
Time:                        21:19:21   Log-likelihood                -1.166e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      49.013
Entities:                         192   P-value                           0.0000
Avg Obs:                       334.34   Distribution:                 F(4,63998)
Min Obs:                       5.0000                                           
Max Obs:                       1137.0

In [12]:
print("=" * 65)
print(f"  Model 10: Distributed-Lag FE (L={L}) — clustered (entity + time)")
print("=" * 65)

df_dl  = df[[ENTITY, DATE, Y] + X9].dropna().copy()
 
date_x = (
    df_dl[[DATE] + X9]
    .drop_duplicates(subset=DATE)
    .sort_values(DATE)
    .reset_index(drop=True)
    .copy()
)
 
X10 = []
for var in X9:
    for lag in range(L + 1):
        lag_name = f"{var}_lag{lag}"
        date_x[lag_name] = date_x[var].shift(lag)
        X10.append(lag_name)
 
data_dl = (
    df_dl[[ENTITY, DATE, Y]]
    .merge(date_x[[DATE] + X10], on=DATE, how="left")
    .dropna(subset=[Y] + X10)
    .set_index([ENTITY, DATE])
    .sort_index()
)
 
m10   = PanelOLS(data_dl[Y], data_dl[X10], entity_effects=True, time_effects=False)
res10 = m10.fit(cov_type="clustered", cluster_entity=True, cluster_time=True)
print(res10.summary)
 
print("=" * 65)
print(f"  Model 10b: Distributed-Lag FE (L={L}) — Driscoll-Kraay (Bartlett, bw=7)")
print("=" * 65)
 
m10b   = PanelOLS(data_dl[Y], data_dl[X10], entity_effects=True, time_effects=False)
res10b = m10b.fit(cov_type="kernel", kernel="bartlett", bandwidth=7)
print(res10b.summary)

  Model 10: Distributed-Lag FE (L=7) — clustered (entity + time)
                          PanelOLS Estimation Summary                           
Dep. Variable:            log_outflow   R-squared:                        0.0086
Estimator:                   PanelOLS   R-squared (Between):             -0.1045
No. Observations:               64187   R-squared (Within):               0.0086
Date:                Wed, May 27 2026   R-squared (Overall):             -0.0684
Time:                        21:19:21   Log-likelihood                -1.164e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      17.434
Entities:                         192   P-value                           0.0000
Avg Obs:                       334.31   Distribution:                F(32,63963)
Min Obs:                       5.0000                                           
Max Obs:                       1137.0   F-st

In [13]:
X11 = [
    "delta_dgs10_z",
    "ads_z",
    "eth_shock",
]

print("=" * 65)
print("  Model 11: VNQ, 1 month treasury, Mortgage Index Returns, VIX + ETH returns excluded")
print("=" * 65)
 
data_m11 = (
    df[[ENTITY, DATE, Y] + X11]
    .dropna()
    .set_index([ENTITY, DATE])
)
 
m11   = PanelOLS(data_m11[Y], data_m11[X11], entity_effects=True, time_effects=False)
res11 = m11.fit(cov_type="clustered", cluster_entity=True, cluster_time=True)
print(res11.summary)
 
print("=" * 65)
print("  Model 11b: VNQ, 1 month treasury, Mortgage Index Returns, VIX + ETH returns excluded (Driscoll-Kraay)")
print("=" * 65)
 
m11b   = PanelOLS(data_m11[Y], data_m11[X11], entity_effects=True, time_effects=False)
res11b = m11b.fit(cov_type="kernel", kernel="bartlett", bandwidth=7)
print(res11b.summary)

  Model 11: VNQ, 1 month treasury, Mortgage Index Returns, VIX + ETH returns excluded
                          PanelOLS Estimation Summary                           
Dep. Variable:            log_outflow   R-squared:                        0.0030
Estimator:                   PanelOLS   R-squared (Between):             -0.0257
No. Observations:               64194   R-squared (Within):               0.0030
Date:                Wed, May 27 2026   R-squared (Overall):             -0.0176
Time:                        21:19:23   Log-likelihood                -1.166e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      64.491
Entities:                         192   P-value                           0.0000
Avg Obs:                       334.34   Distribution:                 F(3,63999)
Min Obs:                       5.0000                                           
Max Obs:               

In [14]:
print("=" * 65)
print(f"  Model 12: Distributed-Lag FE (L={L}) — clustered (entity + time)")
print("=" * 65)

df_dl  = df[[ENTITY, DATE, Y] + X11].dropna().copy()
 
date_x = (
    df_dl[[DATE] + X11]
    .drop_duplicates(subset=DATE)
    .sort_values(DATE)
    .reset_index(drop=True)
    .copy()
)
 
X12 = []
for var in X11:
    for lag in range(L + 1):
        lag_name = f"{var}_lag{lag}"
        date_x[lag_name] = date_x[var].shift(lag)
        X12.append(lag_name)
 
data_dl = (
    df_dl[[ENTITY, DATE, Y]]
    .merge(date_x[[DATE] + X12], on=DATE, how="left")
    .dropna(subset=[Y] + X12)
    .set_index([ENTITY, DATE])
    .sort_index()
)
 
m12   = PanelOLS(data_dl[Y], data_dl[X12], entity_effects=True, time_effects=False)
res12 = m12.fit(cov_type="clustered", cluster_entity=True, cluster_time=True)
print(res12.summary)
 
print("=" * 65)
print(f"  Model 12b: Distributed-Lag FE (L={L}) — Driscoll-Kraay (Bartlett, bw=7)")
print("=" * 65)
 
m12b   = PanelOLS(data_dl[Y], data_dl[X12], entity_effects=True, time_effects=False)
res12b = m12b.fit(cov_type="kernel", kernel="bartlett", bandwidth=7)
print(res12b.summary)

  Model 12: Distributed-Lag FE (L=7) — clustered (entity + time)
                          PanelOLS Estimation Summary                           
Dep. Variable:            log_outflow   R-squared:                        0.0080
Estimator:                   PanelOLS   R-squared (Between):             -0.0876
No. Observations:               64187   R-squared (Within):               0.0080
Date:                Wed, May 27 2026   R-squared (Overall):             -0.0578
Time:                        21:19:24   Log-likelihood                -1.165e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      21.402
Entities:                         192   P-value                           0.0000
Avg Obs:                       334.31   Distribution:                F(24,63971)
Min Obs:                       5.0000                                           
Max Obs:                       1137.0   F-st

10 year treasury yields, ADS and ETH shock are significant

In [15]:
from scipy import stats
 
def cumulative_beta_summary(res, x_vars, label, L=7):
    rows = []
    for var in x_vars:
        lag_names = [f"{var}_lag{l}" for l in range(L + 1)]
        lag_names = [ln for ln in lag_names if ln in res.params.index]
        if not lag_names:
            continue
 
        beta_sum   = res.params[lag_names].sum()
        cov_sub    = res.cov.loc[lag_names, lag_names]
        se_sum     = np.sqrt(cov_sub.values.sum())
        t_stat     = beta_sum / se_sum
        p_value    = 2 * (1 - stats.norm.cdf(abs(t_stat)))
        pct_effect = 100 * (np.exp(beta_sum) - 1)
        sig        = "***" if p_value < 0.01 else "**" if p_value < 0.05 else "*" if p_value < 0.10 else ""
 
        rows.append({
            "variable":   var,
            "cumul_beta":   round(beta_sum,   4),
            "std_err":    round(se_sum,     4),
            "t_stat":     round(t_stat,     3),
            "p_value":    round(p_value,    4),
            "sig":        sig,
            "pct_effect": round(pct_effect, 2),
        })
 
    df_out = pd.DataFrame(rows)
    print(f"=== Cumulative IRF — {label} ===")
    print(f"(sum of lag0..lag{L} per variable, SE via delta method)")
    print(df_out.to_string(index=False))
    print()
    return df_out
 
 
# Models 2 / 2b 
cumulative_beta_summary(res2,  X1, "Model 2  — Clustered")
cumulative_beta_summary(res2b, X1, "Model 2b — Driscoll-Kraay")
 
# Models 4 / 4b 
cumulative_beta_summary(res4,  X3, "Model 4  — Clustered")
cumulative_beta_summary(res4b, X3, "Model 4b — Driscoll-Kraay")
 
# Models 6 / 6b
cumulative_beta_summary(res6,  X5, "Model 6  — Clustered")
cumulative_beta_summary(res6b, X5, "Model 6b — Driscoll-Kraay")
 
# Models 8 / 8b 
cumulative_beta_summary(res8,  X7, "Model 8  — Clustered")
cumulative_beta_summary(res8b, X7, "Model 8b — Driscoll-Kraay")
 
# Models 10 / 10b  
cumulative_beta_summary(res10,  X9, "Model 10  — Clustered")
cumulative_beta_summary(res10b, X9, "Model 10b — Driscoll-Kraay")
 
# Models 12 / 12b 
cumulative_beta_summary(res12,  X11, "Model 12  — Clustered")
cumulative_beta_summary(res12b, X11, "Model 12b — Driscoll-Kraay")

=== Cumulative IRF — Model 2  — Clustered ===
(sum of lag0..lag7 per variable, SE via delta method)
      variable  cumul_beta  std_err  t_stat  p_value sig  pct_effect
   delta_vix_z     -0.0762   0.0431  -1.767   0.0772   *       -7.34
 delta_dgs10_z     -0.1619   0.0361  -4.488   0.0000 ***      -14.95
delta_dgs1mo_z      0.0164   0.0405   0.405   0.6856            1.65
         ads_z     -0.0599   0.0206  -2.903   0.0037 ***       -5.81
  eth_return_z     -0.0866   0.0615  -1.408   0.1590           -8.29
  vnq_return_z     -0.1514   0.0565  -2.680   0.0074 ***      -14.05
mortgage_ret_z      0.0275   0.0486   0.566   0.5717            2.78
     eth_shock     -0.5689   0.0981  -5.801   0.0000 ***      -43.38

=== Cumulative IRF — Model 2b — Driscoll-Kraay ===
(sum of lag0..lag7 per variable, SE via delta method)
      variable  cumul_beta  std_err  t_stat  p_value sig  pct_effect
   delta_vix_z     -0.0762   0.0678  -1.125   0.2608           -7.34
 delta_dgs10_z     -0.1619   0.0629

,variable,cumul_beta,std_err,t_stat,p_value,sig,pct_effect
0,delta_dgs10_z,-0.1166,0.0598,-1.949,0.0513,*,-11.01
1,ads_z,-0.0572,0.0329,-1.738,0.0823,*,-5.56
2,eth_shock,-0.4616,0.1072,-4.307,0.0000,***,-36.97
